# 08 Link text reports to satellite tiles (by event and time window)

The text side (HumAID tweets, notebooks 01 to 06) and the image side (tile evidence, verified claims, advisory) are separate. This notebook puts them side by side for one tile.

**What the link is.** A tile's xBD event is matched to the HumAID event of the same disaster. The tweets shown are that event's highest-scoring tweets from the notebook 06 usefulness score, optionally limited to a time window around the image date.

**What it is not.** HumAID tweets carry no coordinates, so a tweet is linked to the event, not to the place on the tile. Say "linked by event and time window, not by location" in the report. Only four xBD events have a HumAID counterpart (Harvey, Florence, Matthew, Midwest floods). Nepal has none, and the notebook says so.

**Privacy and licence.** Tweet text stays in this notebook: @handles and links are masked, and the saved file holds tweet ids and scores only unless you set `SAVE_TEXT = True`. Check the HumAID licence before redistributing any text, and do not commit outputs.

Runs on CPU (scoring about 3,700 tweets takes a few minutes) or GPU. The advisory part is optional (`INCLUDE_ADVISORY`).

## 1. Setup

In [ ]:
import os, sys, json, shutil, subprocess
from google.colab import drive
drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
REPO = "/content/FloodVerify"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/aaarrvind/FloodVerify", REPO], check=True)
NEW_GENERATE = f"{PROJECT}/advisory_generation/generate.py"      # only until the generate.py pull request is merged
if os.path.exists(NEW_GENERATE):
    shutil.copy(NEW_GENERATE, f"{REPO}/src/floodsitrep/advisory/generate.py")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "transformers", "pyarrow", "sentence-transformers"], check=True)
if f"{REPO}/src" not in sys.path:
    sys.path.insert(0, f"{REPO}/src")

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
from floodsitrep.nlp.claims import report_from_cells
from floodsitrep.nlp.checker import verify
from floodsitrep.common.interfaces import load_evidence
from floodsitrep.pipeline.run import run_tile

HUMAID = f"{PROJECT}/humaid"
MODEL_DIR = f"{HUMAID}/transformer"                  # saved by notebook 02
EVENTS_PARQUET = f"{HUMAID}/events.parquet"          # saved by notebook 05
CAPTIONS_FILE = f"{PROJECT}/xbd/processed/captions/qwen3vl4b_cells_50.json"
EVIDENCE_DIR = f"{PROJECT}/evidence_predicted/m1_siamese_unet_ce_s0"     # predicted evidence (only some events have it)
LABEL_EVIDENCE_DIR = f"{PROJECT}/xbd/processed/evidence_labels"          # evidence from xBD labels, used when a tile has no predicted evidence
OUT = f"{PROJECT}/advisory_generation"

XBD_TO_HUMAID = {"hurricane-harvey": "hurricane_harvey_2017", "hurricane-florence": "hurricane_florence_2018",
                 "hurricane-matthew": "hurricane_matthew_2016", "midwest-flooding": "midwestern_us_floods_2019"}
ACTIONABLE = ["infrastructure_and_utility_damage", "injured_or_dead_people", "displaced_people_and_evacuations",
              "requests_or_urgent_needs", "missing_or_found_people", "caution_and_advice"]

THETA = 0.75            # usefulness threshold from notebook 06 (a number between 0 and 1); None shows the top k without a cut-off
TOP_K = 5               # tweets shown per tile
CAPTURE_DATES = {}      # optional {"hurricane-harvey": "2017-09-02", ...}: image date per xBD event, to limit tweets to a window
WINDOW_DAYS = 7         # tweets within this many days of the capture date are kept
INCLUDE_ADVISORY = True    # True also loads the retrieval model and prints the advisory (needs passages in guidance/)
SAVE_TEXT = False           # False: the saved file holds tweet ids and scores, not tweet text
device = "cuda" if torch.cuda.is_available() else "cpu"
CAPTIONS = json.load(open(CAPTIONS_FILE))
print("device:", device)

## 2. Functions
Build and compute functions are separate from the display cell.

In [ ]:
def xbd_event(tile_id):
    """xBD event name of a tile id such as hurricane-harvey_00000183_p3."""
    return tile_id.rsplit("_", 2)[0]


def snowflake_time(tweet_id):
    """Posting time encoded in a tweet id (UTC), or NaT when the id is missing."""
    n = pd.to_numeric(tweet_id, errors="coerce")
    if pd.isna(n):
        return pd.NaT
    return pd.to_datetime((int(n) >> 22) + 1288834974657, unit="ms", utc=True)


def mask_text(text):
    """Tweet text with @handles and links replaced, so no account name or URL is shown or saved."""
    text = re.sub(r"https?://\S+", "<link>", str(text))
    return re.sub(r"@\w+", "@user", text)


def score_tweets(texts, model, tokenizer, labels, actionable, batch=64, max_len=128):
    """(probabilities, usefulness score) per text. The score is the probability mass on the actionable classes."""
    out = []
    texts = list(texts)
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch):
            enc = tokenizer(texts[i:i + batch], truncation=True, max_length=max_len, padding=True, return_tensors="pt").to(device)
            out.append(torch.softmax(model(**enc).logits.float(), dim=-1).cpu().numpy())
    probs = np.vstack(out)
    idx = [labels.index(c) for c in actionable]
    return probs, probs[:, idx].sum(axis=1)


def scored_events(events, events_wanted, model, tokenizer, labels, actionable):
    """Test-split tweets of the wanted HumAID events with class, score and posting time."""
    df = events[(events["event"].isin(events_wanted)) & (events["split"] == "test")].reset_index(drop=True)
    probs, score = score_tweets(df["text"], model, tokenizer, labels, actionable)
    df["predicted_class"] = np.array(labels)[probs.argmax(axis=1)]
    df["score"] = score
    df["posted"] = [snowflake_time(t) for t in df["tweet_id"]]
    return df


def tweets_for_tile(tile_id, scored, k=None, theta=None, capture_dates=None, window_days=None):
    """The k highest-scoring tweets of the tile's HumAID event, at or above theta, inside the time window if one is set.
    Settings left as None are read from the settings cell when the function runs, so changing them there takes effect.
    Returns (DataFrame, note); the DataFrame is empty when the event has no HumAID counterpart."""
    k = TOP_K if k is None else k
    theta = THETA if theta is None else theta
    capture_dates = CAPTURE_DATES if capture_dates is None else capture_dates
    window_days = WINDOW_DAYS if window_days is None else window_days
    xe = xbd_event(tile_id)
    he = XBD_TO_HUMAID.get(xe)
    if he is None:
        return pd.DataFrame(), f"no HumAID event for {xe}"
    d = scored[scored["event"] == he]
    note = f"{len(d)} test tweets of {he}"
    if xe in capture_dates:
        centre = pd.Timestamp(capture_dates[xe], tz="UTC")
        d = d[(d["posted"] - centre).abs() <= pd.Timedelta(days=window_days)]
        note += f"; {len(d)} within {window_days} days of {capture_dates[xe]}"
    if theta is not None:
        d = d[d["score"] >= theta]
        note += f"; {len(d)} at or above threshold {theta}"
    d = d.sort_values("score", ascending=False, kind="stable").head(k)
    d = d.assign(rank=range(1, len(d) + 1), text=d["text"].map(mask_text))
    return d[["rank", "score", "predicted_class", "posted", "tweet_id", "text"]], note


def pick_tiles_per_event(captions, evidence_dirs, events=tuple(XBD_TO_HUMAID), require_supported=True):
    """For each xBD event with a HumAID counterpart: (tile_id, evidence_dir) of the first captioned tile with a supported claim.
    evidence_dirs are tried in order (predicted first, labels second), so a tile uses predicted evidence when it exists.
    With require_supported=False a tile with evidence but no supported claim is accepted."""
    found = {}
    for d in evidence_dirs:
        for tid in captions:
            e = xbd_event(tid)
            path = f"{d}/{tid}.json"
            if e not in events or e in found or not os.path.exists(path):
                continue
            claims = verify(report_from_cells(captions[tid]), load_evidence(path))
            if not require_supported or any(c["verdict"] == "supported" for c in claims):
                found[e] = (tid, d)
    return found

## 3. Load the classifier and score the tweets
Only test-split tweets are used, so the model has not seen them in training.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device).eval()
LABELS = [model.config.id2label[i] for i in range(model.config.num_labels)]
assert all(c in LABELS for c in ACTIONABLE), "actionable classes missing from the model labels"

events = pd.read_parquet(EVENTS_PARQUET)
scored = scored_events(events, list(XBD_TO_HUMAID.values()), model, tokenizer, LABELS, ACTIONABLE)
print(len(scored), "test tweets scored")
print(scored.groupby("event").agg(tweets=("text", "size"), first=("posted", "min"), last=("posted", "max")))

## 4. Coverage
Which tiles can be linked. The posting-time columns above show when each event's tweets were posted. If they fall far from the image date, set `CAPTURE_DATES` (the capture date of the xBD image for that event, from the xBD label metadata) and `WINDOW_DAYS`.

In [ ]:
tiles = pd.Series([xbd_event(t) for t in CAPTIONS]).value_counts().rename("captioned tiles").to_frame()
tiles["HumAID event"] = [XBD_TO_HUMAID.get(e, "none") for e in tiles.index]
tiles

## 5. One tile per event: verified claims and linked tweets

In [ ]:
def show_linked(tile_id, scored, evidence_dir=None):
    """Print the verified claims of a tile and the tweets linked to its event."""
    evidence_dir = EVIDENCE_DIR if evidence_dir is None else evidence_dir
    kind = "predicted evidence" if evidence_dir == EVIDENCE_DIR else "evidence from xBD labels (not predicted)"
    out = run_tile(tile_id, evidence_dir=evidence_dir, report_text=report_from_cells(CAPTIONS[tile_id]))
    print("=" * 90, "\nTILE", tile_id, "|", kind)
    for c in out["claims"] or []:
        print(f"  {c['verdict']:<14} {c.get('cell')}: {c['text']}")
    tw, note = tweets_for_tile(tile_id, scored)
    print("\nLinked text reports (by event and time window, not by location):", note)
    for _, r in tw.iterrows():
        print(f"  {r['rank']}. [{r['score']:.3f}] {r['predicted_class']}: {r['text']}")
    return out, tw


chosen = pick_tiles_per_event(CAPTIONS, [EVIDENCE_DIR, LABEL_EVIDENCE_DIR])
print("tiles:", chosen)
missing = [e for e in XBD_TO_HUMAID if e not in chosen]
if missing:
    print("no tile with a supported claim for:", missing)
results = {e: show_linked(t, scored, evidence_dir=d) for e, (t, d) in chosen.items()}

## 6. Optional: the advisory next to the tweets
Set `INCLUDE_ADVISORY = True` at the top and rerun this cell. It uses the retrieval model and the guidance passages, as in notebook 07 (mode B, passages with ids).

In [ ]:
if INCLUDE_ADVISORY:
    from floodsitrep.advisory import retrieve as R
    from floodsitrep.advisory.generate import set_backends, make_retriever
    G = f"{PROJECT}/guidance"
    passages = R.load_passages(f"{G}/passages.csv")
    emb_model = R.load_model()
    vectors = R.load_or_embed(emb_model, passages, f"{G}/passages_bge_m3.npy")
    set_backends(retrieve_fn=make_retriever(emb_model, vectors, passages))
    for e, (t, d) in chosen.items():
        out = run_tile(t, evidence_dir=d, report_text=report_from_cells(CAPTIONS[t]))
        print(t, "\n", out["advisory"]["en"], "\n")
else:
    print("advisory skipped (INCLUDE_ADVISORY = False)")

## 7. Save
Tweet ids and scores only, unless `SAVE_TEXT = True`.

In [ ]:
def linked_table(results, save_text=None):
    """One row per linked tweet with its tile; tweet text only when save_text is True."""
    save_text = SAVE_TEXT if save_text is None else save_text
    rows = []
    for e, (out, tw) in results.items():
        for _, r in tw.iterrows():
            row = {"tile_id": out["tile_id"], "xbd_event": e, "rank": r["rank"], "score": round(float(r["score"]), 3),
                   "predicted_class": r["predicted_class"], "tweet_id": r["tweet_id"], "posted": str(r["posted"])}
            if save_text:
                row["text"] = r["text"]
            rows.append(row)
    return pd.DataFrame(rows)


os.makedirs(OUT, exist_ok=True)
linked = linked_table(results)
linked.to_csv(f"{OUT}/linked_tweets_demo.csv", index=False)
print(len(linked), "rows saved to", f"{OUT}/linked_tweets_demo.csv")